# 03 · Behaviour

**What this notebook does.** It describes how the rats behaved, using the kept trials from 01: whether errors change the next
trial (B1–B3), whether the trial labels are consistent, and how correct, early and late responses change **along the session**.
The session phases defined here (start / middle / end) are used again in 07, section 4.

**Run order in the project:** 01 → 02 → **03** → 04 → 05 → 06 → 07. Needs `trials_filtered.csv` and `final_sessions.csv` from 01.

**Steps**

| Step | Question | Main test | Output |
|---|---|---|---|
| 1 | **B1** Are correct responses slower after an error? (post-error slowing) | mixed model on RT | `B1_post_error_slowing_lmm.csv`, `B1_post_error_slowing.csv` |
| 2 | **B2** Is the next trial more often an early release after an error? | per-session Wilcoxon + GEE | `B2_early_after_error.csv`, `B2_early_after_error_gee.csv` |
| 3 | **B3** Do rats press faster before early than before correct trials? | per-session Wilcoxon + mixed model | `B3_latency_to_press.csv`, `B3_latency_to_press_lmm.csv` |
| 4 | Are the trial labels consistent? | checks | – |
| 5 | Correct / early / late along the session, and the phase borders | sign-flip test | `outcome_share_phases.csv` + figure |
| 6 | Number of correct / early / late per rat, trial type and phase | counts | 3 tables + 3 figures |
| 7 | Summary of the results | – | – |

**How to use it:** run the settings cell first; then any cell. Nothing is written unless `SAVE = True`.
Change things only in the settings cell. Functions: `fx_behaviour.py`; colours and saving: `fx_common.py`.

## Settings

**Paths:** `CODE_DIR`, `DATA_ROOT`, `AMIR`, `TABLES` (as in 01; `TABLES` temporarily the old folder), `FIG_DIR` (figures of this notebook).

**Saving:** `SAVE = False` writes nothing; `True` writes figures to `FIG_DIR` and tables to `TABLES`. To save one cell elsewhere,
add `folder=r"...", force=True` to its save line.

**Data loaded here** (so every cell can run alone):
- `TR`: all trials of the 60 final sessions (`trials_filtered.csv`).
- `K`: the **kept** trials (`keep == True`, rules R1–R5 of 01), with the session flags `single_wt_session` and `short_vs_long`.
  Columns used: `label` (correct/early/late), `type` (short/long), `rt_s` = release − tone, `prev_outcome` = label of the previous
  trial (computed before filtering), `latency_to_press_s` = end of the previous trial → press, `long_prob`.

**Session phases (steps 5–6):** `PHASE_CUTS = (0.25, 0.85)`: position in the session 0–0.25 = start (warm-up), 0.25–0.85 = middle
(plateau), 0.85–1 = end (decline). Position = rank of the trial in its session / number of trials (0 = first, 1 = last), computed on
all trials before removing non-kept ones. `N_BINS = 20` bins for the curves. `BLOCKS_AS_BURSTS = False` uses all `long_prob` blocks;
`True` only 0.5 and ≥ 0.75 blocks (as an older burst analysis).

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
warnings.simplefilter("ignore")

# ── paths ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server
RAW       = os.path.join(DATA_ROOT, "0_raw")                                       # raw recordings (ap.bin)
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings + share/trials.csv
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder (all outputs)
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
FIG_DIR   = os.path.join(AMIR, "figures", "03_behaviour")                          # figures of this notebook (if SAVE)

# ── saving ──
SAVE = False                                                                       # True = write figures and tables

# ── session phases (steps 5–6) ──
PHASE_CUTS       = (0.25, 0.85)                                                    # start | middle | end borders (position 0–1)
N_BINS           = 20                                                              # bins for the curves along the session
BLOCKS_AS_BURSTS = False                                                           # True = only 0.5 and >= 0.75 long_prob blocks

# ── code + data ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, fx_behaviour as fb
importlib.reload(fxc); importlib.reload(fb)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
fxc.check_paths(TABLES=TABLES)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)

final = pd.read_csv(os.path.join(TABLES, "final_sessions.csv"), dtype={"rat": str})
TR    = pd.read_csv(os.path.join(TABLES, "trials_filtered.csv"), dtype={"rat": str}, low_memory=False)
for c in ["label", "type", "rt_s", "prev_outcome", "latency_to_press_s", "long_prob", "keep"]:
    assert c in TR.columns, f"trials_filtered.csv has no column '{c}' - rerun the trial filter in 01"
K = TR[TR["keep"]].merge(final[["session", "single_wt_session", "short_vs_long"]], on="session", how="left")
print(f"kept trials: {len(K)} (expected 8615) | sessions: {K['session'].nunique()} (expected 60)")
print(pd.crosstab(K["rat"], [K["type"], K["label"]], margins=True).to_string())

## Step 1 · B1 Post-error slowing (Narayanan & Laubach 2008)

**Question.** Are correct responses slower when the previous trial was an error?

**Trials.** Kept **correct** trials whose previous trial was correct, early or late (touch, other and the first trial of a
session are left out). Short and long trials are pooled, with trial type as a covariate.

**Main test: linear mixed model** (one per rat and one pooled):
`RT (ms) ~ after_late + after_early + long (+ rat in the pooled model)`, random intercept per session, REML.
- `after_late`, `after_early` = 1 if the previous trial was a late / early error; the reference is "after correct".
- Why a mixed model: it uses every trial while accounting for sessions differing in their overall RT (random intercept), and it
  separates the two kinds of error and the trial type in one model. Wald p-values, raw.

**Check: per-session Wilcoxon.** For each session (≥ 5 trials in both groups): median RT after correct and after an error
(early or late together); Wilcoxon signed-rank test across sessions, per rat and pooled. It treats the session as the unit.

**Result (2026-10-23):** after a late error **+35 ms** [27, 44] (rat 134 +47, 135 +36, 136 +32; all p < 1e-5); after an early error
+9 ms [2, 16], p = 0.015, only rat 134 alone (+29 ms, p = 0.0006). Wilcoxon: +12 ms, 39/58 sessions, p = 0.004.

**Saves:** `B1_post_error_slowing_lmm.csv`, `B1_post_error_slowing.csv`.

In [ ]:
B1_lmm, B1_wilcoxon = fb.b1_post_error_slowing(K)
fxc.save_table(B1_lmm, "B1_post_error_slowing_lmm"); fxc.save_table(B1_wilcoxon, "B1_post_error_slowing")

## Step 2 · B2 Early releases after an error (Donnelly et al. 2014)

**Question.** Is the next trial more often an early release after an error?

**Trials.** Kept correct, early and late trials whose previous trial was correct, early or late.

**Main test: per-session Wilcoxon.** For each session (≥ 10 trials in both groups): the share of early releases after a correct
trial and after an error (early or late together); Wilcoxon signed-rank across sessions, per rat and pooled.

**Check: trial-level logistic GEE.** `early ~ after_late + after_early + long (+ rat)`, sessions as clusters with exchangeable
correlation; the result is an **odds ratio** against "after correct" (< 1 = fewer early releases). Why GEE: the outcome is yes/no
per trial, and trials of one session are not independent.

**Result (2026-10-23):** after a late error **fewer** early releases, OR 0.70, p = 0.0002 (rats 135: 0.64, 136: 0.65; 134 n.s.);
after an early error no change (OR 0.91, p = 0.24). Wilcoxon (both errors together): −2.4 %-points, 20/57 sessions, p = 0.052.

**Saves:** `B2_early_after_error.csv`, `B2_early_after_error_gee.csv`.

In [ ]:
B2_wilcoxon, B2_gee = fb.b2_early_after_error(K)
fxc.save_table(B2_wilcoxon, "B2_early_after_error"); fxc.save_table(B2_gee, "B2_early_after_error_gee")

## Step 3 · B3 Latency to press before early vs correct trials (Donnelly et al. 2014)

**Question.** Do rats press sooner (shorter pause) before trials that end in an early release?

**Measure.** Latency to press = end of the previous trial → this press (behaviour clock, computed before filtering). Kept correct and
early trials; latencies ≤ 0 are removed.

**Main test: per-session Wilcoxon** of the median latency before correct vs before early (≥ 5 trials each).

**Check: mixed model** on `log10(latency) ~ early + long (+ rat)`, random intercept per session. The log is used because latencies
are strongly skewed (a few long pauses). The coefficient is turned back into a **ratio** early / correct (< 1 = faster before early).

**Result (2026-10-23):** no faster pressing before early trials: Wilcoxon p = 0.54; model ratio 1.04 [1.00, 1.08], p = 0.039
(slightly *longer*, opposite to Donnelly 2014; no rat alone significant).

**Saves:** `B3_latency_to_press.csv`, `B3_latency_to_press_lmm.csv`.

In [ ]:
B3_wilcoxon, B3_lmm = fb.b3_latency_to_press(K)
fxc.save_table(B3_wilcoxon, "B3_latency_to_press"); fxc.save_table(B3_lmm, "B3_latency_to_press_lmm")

## Step 4 · Label checks

**What it checks** (1–3 on all trials of the final sessions, 4–7 on kept trials):
1. `hold_duration` in `trials.csv` equals release − press.
2. Touch vs early: every touch is also labelled early in `trials.csv`; touch = hold < 0.1 s; no early trial (our label) with hold < 0.1 s.
3. Label combinations per trial: the only double label should be touch + early.
4. Response window: correct RT 0–0.5 s, late RT > 0.5 s, and correct and late never overlap within a session.
5. `full_hold`: kept early trials all have `full_hold == 0`.
6. Press time from the TTL vs from `press_np_s`, per session.
7. Very fast correct responses (RT < 0.12 s).

**Expected (2026-10-23):** all consistent after the filter; all 656 touches labelled early; kept correct RT 0.01–0.50 s, late 0.51–8.56 s,
no overlap; **269 of 4,486 correct trials (6 %) have RT < 0.12 s**. These are probably releases timed in advance rather than reactions
to the tone; they are kept (the rule "correct RT ≥ 0.12 s" was removed on 2026-10-22).

In [ ]:
fb.label_checks(TR, K)

## Step 5 · Correct, early and late along the session, and the session phases

**What it does.** Long trials only (correct + early + late), all `long_prob` blocks (unless `BLOCKS_AS_BURSTS = True`).
- **Position** of each trial in its session (0 = first, 1 = last).
- **Curves:** for each session and each of `N_BINS` position bins, the % correct, % early and % late (each of all three outcomes);
  then mean ± SEM over sessions, for all rats and each rat.
- **Phases:** borders at `PHASE_CUTS` (0.25, 0.85) → start (warm-up), middle (plateau), end (decline). Per session and phase: % of each outcome.
- **Test:** for each outcome, start − middle and end − middle per session, **sign-flip test** across sessions (each session's difference
  randomly flipped in sign 10,000 times; two-sided, raw p). Why: it only assumes the differences are symmetric around 0 under no
  effect, and the session is the unit.

**Figure:** rows = % correct, % early, % late, and all three together; columns = all rats and each rat; dashed lines = phase borders;
above each phase the % and the trial numbers (outcome / other trials).

**Saves:** `outcome_share_phases.csv`, figure `outcome share along the session.png`.

In [ ]:
outcome_phases, long_trials = fb.outcome_share_phases(TR, PHASE_CUTS, N_BINS, BLOCKS_AS_BURSTS)
fig = fb.plot_outcome_share_phases(outcome_phases, long_trials, PHASE_CUTS, N_BINS)
fxc.save_table(outcome_phases, "outcome_share_phases"); fxc.save_fig(fig, "outcome share along the session"); plt.show()

## Step 6 · Number of correct, early and late trials per rat, trial type and phase

**What it does.** Kept trials of the final sessions, all blocks. Counts and % of correct / early / late (of correct + early + late):
- **A** whole session, per rat and trial type (long, short);
- **B** per session phase (same borders as step 5), per rat and trial type;
- **C** per phase, long + short and all rats pooled.

Descriptive only (no test). Figures: stacked bars, the % and the number of trials written inside each part.

**Saves:** `outcome_counts_per_rat.csv`, `outcome_counts_phases.csv`, `outcome_counts_phases_pooled.csv` and the figures
`outcome counts per rat.png`, `outcome counts per phase.png`, `outcome counts per phase pooled.png`.

In [ ]:
counts_rat, counts_phase, counts_pooled, n_sess = fb.outcome_counts(TR, final, PHASE_CUTS)
figA, figB, figC = fb.plot_outcome_counts(counts_rat, counts_phase, counts_pooled, n_sess)
fxc.save_table(counts_rat, "outcome_counts_per_rat"); fxc.save_table(counts_phase, "outcome_counts_phases")
fxc.save_table(counts_pooled, "outcome_counts_phases_pooled", index=True)
for f_, n_ in ((figA, "outcome counts per rat"), (figB, "outcome counts per phase"), (figC, "outcome counts per phase pooled")): fxc.save_fig(f_, n_)
plt.show()

## Step 7 · Summary of the behaviour part (2026-10-23)

**Data:** 63 usable sessions → 60 final (3 WT-manipulation sessions of rat 134 excluded): rat 134: 11, 135: 29, 136: 20.
7 single-WT sessions (only 1.0 s: rat 135 ×6, rat 134 ×1) kept, but not used for short vs long (53 sessions).
All `long_prob` blocks kept. 9 sessions with `sync_ok` False kept (trials without a neural time are removed by R1).

**Definitions**
- label: touch (hold < 0.1 s, checked first) / correct (release 0–0.5 s after the tone) / early (before the tone) / late (> 0.5 s after the tone) / other
- type from `wt` (0.3 short, 1.0 long), not from the `long` column
- neural clock: press = `press_ttl_np_s`, else `press_np_s`; tone and release = press + their behaviour-clock delay
- `rt_s` = release − tone; `prev_outcome` = label of the previous trial; `latency_to_press_s` = previous trial end → press

**Trial funnel (01):** 9,877 → R1 no press time → R2 touch −633 → R3 early held > 1.05 s −0 → R4 other −30 → R5 timing mismatch −222
→ **8,615 kept** (rat 134: 1,504, 135: 4,659, 136: 2,452). Press from TTL 7,851, from `press_np_s` 764.

**Behaviour results**
- **B1 post-error slowing:** after a late error +35 ms [27, 44] (all rats, p < 1e-5); after an early error +9 ms, p = 0.015, only rat 134.
- **B2 early releases after an error:** after a late error **fewer** early releases (OR 0.70, p = 0.0002; rats 135, 136); after an early error no change.
- **B3 latency to press:** no faster pressing before early trials (Wilcoxon p = 0.54; model ratio 1.04, p = 0.039, slightly longer).
- Together: after a late error the rats hold longer (slower correct responses and fewer early releases); after an early error only rat 134 slows down.

**Open questions (supervisor / pipeline author):** how the task program decides correct vs late; whether its delay is the same for press
and release; the rule behind `sync_ok`; how press TTLs are matched to trials outside the sync span; how `channel_regions.csv` was made
(tilt = 0); task version per session (reward-size unit); how `long_prob` blocks switch.